<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    09 · Modelos QSAR: Evaluación · Interpretación
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 4 — Del dato curado al modelo predictivo</em>
  </p>
</div>


---
## ¿Qué es QSAR y para qué sirve?

**QSAR** (Quantitative Structure-Activity Relationship — Relación Cuantitativa Estructura-Actividad)  
es el enfoque computacional que conecta la **estructura química** de una molécula  
con su **actividad biológica** medida experimentalmente.

La idea central es simple:

> *Moléculas con estructuras similares tienden a tener actividades similares.*

Un modelo QSAR aprende esa relación desde datos históricos y luego predice  
la actividad de moléculas **nunca antes ensayadas** — ahorrando tiempo, dinero y recursos.

### ¿Qué haremos en este notebook?

| # | Sección | Modelos / conceptos |
|---|---------|-------------------|
| 1 | Carga y preparación del dataset | Features de NB-DATA-03 |
| 2 | Baseline: modelo trivial | Siempre predecir la clase mayoritaria |
| 3 | Regresión logística | Modelo lineal interpretable |
| 4 | Support Vector Machine (SVM) | Margen máximo, kernel RBF |
| 5 | Random Forest | Ensemble de árboles, importancia de features |
| 6 | Gradient Boosting (XGBoost) | Boosting iterativo |
| 7 | Comparación de modelos | Tabla y gráficos de métricas |
| 8 | Validación cruzada | Estimación robusta del rendimiento |
| 9 | Curvas ROC y Precision-Recall | Evaluación completa |
| 10 | Interpretación: importancia de features | ¿Qué aprende el modelo? |
| 11 | Y-scrambling | ¿El modelo realmente aprendió? |
| 12 | Pipeline completo y guardado | Modelo listo para uso |

---
> **Entrada:** archivos generados en NB-DATA-02 y NB-DATA-03  
> **Salida:** modelo entrenado + reporte de evaluación completo


---
## 1. Instalación y preparación del dataset

In [ ]:
# ── Instalar librerías ──────────────────────────────────────────────────────
!pip install scikit-learn xgboost imbalanced-learn rdkit --quiet

print("✅ Librerías instaladas")


In [ ]:
# ── Instalar mordred si no está disponible ───────────────────────────────────
try:
    from mordred import Calculator, descriptors as mordred_desc
    print("✅ mordred ya instalado")
except ImportError:
    import subprocess, sys
    print("Instalando mordredcommunity (compatible con numpy 2.x)...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "mordredcommunity>=2.0", "--quiet"], check=True)
    from mordred import Calculator, descriptors as mordred_desc
    print("✅ mordredcommunity instalado")

In [ ]:
# ── Importaciones ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, pickle
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, learning_curve)
from sklearn.preprocessing   import StandardScaler, label_binarize
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (roc_auc_score, f1_score, matthews_corrcoef,
                                     accuracy_score, confusion_matrix,
                                     roc_curve, precision_recall_curve,
                                     average_precision_score,
                                     classification_report, ConfusionMatrixDisplay)
from sklearn.dummy           import DummyClassifier
from sklearn.inspection      import permutation_importance

# XGBoost
try:
    import xgboost as xgb
    XGB_OK = True
    print("✅ XGBoost disponible")
except ImportError:
    XGB_OK = False
    print("⚠️  XGBoost no disponible")

# RDKit (para visualización molecular al final)
from rdkit import Chem
from rdkit.Chem import Draw

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
print("✅ Todo listo")


In [ ]:
# ── Cargar los archivos de NB-DATA-02 y NB-DATA-03 ─────────────────────────
# Los archivos se generaron en los notebooks anteriores.
# Si no los tienes, el código descargará un dataset de ejemplo automáticamente.

import os

ARCHIVO_LABELS   = None   # ← Pon aquí tu archivo labels_<target>.csv
ARCHIVO_DESC     = None   # ← Pon aquí tu archivo features_descriptores_<target>.csv
ARCHIVO_FP       = None   # ← Pon aquí tu archivo features_morgan_<target>.npy

# Buscar archivos automáticamente
for f in os.listdir('.'):
    if f.startswith('labels_') and f.endswith('.csv'):
        ARCHIVO_LABELS = f
    if f.startswith('features_descriptores_') and f.endswith('.csv'):
        ARCHIVO_DESC = f
    if f.startswith('features_morgan_') and f.endswith('.npy'):
        ARCHIVO_FP = f

if ARCHIVO_LABELS:
    print(f"✅ Archivos encontrados:")
    print(f"   Labels:       {ARCHIVO_LABELS}")
    print(f"   Descriptores: {ARCHIVO_DESC}")
    print(f"   Fingerprints: {ARCHIVO_FP}")
else:
    print("⚠️  Archivos no encontrados — descargando dataset EGFR de ejemplo...")


In [ ]:
# ── Carga o descarga del dataset ────────────────────────────────────────────
from math import log

def descargar_egfr_demo():
    """Descarga un dataset EGFR de ChEMBL para usar como ejemplo."""
    from chembl_webresource_client.new_client import new_client
    from rdkit.Chem import Descriptors, AllChem
    from rdkit import Chem, DataStructs
    from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect

    activity = new_client.activity
    print("  Descargando datos de EGFR (puede tardar 1 min)...")
    datos = activity.filter(
        target_chembl_id='CHEMBL203',
        standard_type='IC50',
        standard_units='nM',
        assay_type='B'
    ).only('canonical_smiles','molecule_chembl_id','standard_value','pchembl_value')

    df = pd.DataFrame.from_records(datos)
    df = df.astype({'standard_value': float}).dropna()
    df = df[df['standard_value'] > 0]
    df['pActividad'] = [-log(v/1e9, 10) for v in df['standard_value']]
    df = df[(df['pActividad'] >= 5) & (df['pActividad'] <= 12)]
    df = df.drop_duplicates('canonical_smiles').reset_index(drop=True)
    df['activo'] = (df['pActividad'] >= 6).astype(int)
    df = df.rename(columns={'canonical_smiles':'std_smiles','standard_value':'IC50_nM'})

    # Calcular descriptores y fingerprints básicos
    desc_cols = ['MolWt','MolLogP','NumHDonors','NumHAcceptors',
                 'TPSA','NumRotatableBonds','RingCount']
    rows, fps = [], []
    for smi in df['std_smiles']:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            rows.append({
                'MolWt':             Descriptors.MolWt(mol),
                'MolLogP':           Descriptors.MolLogP(mol),
                'NumHDonors':        Descriptors.NumHDonors(mol),
                'NumHAcceptors':     Descriptors.NumHAcceptors(mol),
                'TPSA':              Descriptors.TPSA(mol),
                'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
                'RingCount':         Descriptors.RingCount(mol),
            })
            fp = GetMorganFingerprintAsBitVect(mol, 2, 2048)
            arr = np.zeros(2048, dtype=np.uint8)
            DataStructs.ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        else:
            rows.append({c: np.nan for c in desc_cols})
            fps.append(np.zeros(2048, dtype=np.uint8))

    df_desc = pd.DataFrame(rows)
    df_full = pd.concat([df.reset_index(drop=True), df_desc], axis=1)
    df_full.to_csv('labels_egfr_demo.csv', index=False)
    np.save('features_morgan_egfr_demo.npy', np.array(fps))
    print(f"  ✅ Dataset demo guardado: {len(df_full)} compuestos")
    return df_full, np.array(fps), desc_cols

if ARCHIVO_LABELS:
    df_labels = pd.read_csv(ARCHIVO_LABELS)
    X_fp = np.load(ARCHIVO_FP) if ARCHIVO_FP else None
    df_desc_raw = pd.read_csv(ARCHIVO_DESC) if ARCHIVO_DESC else None
    TARGET_NAME = ARCHIVO_LABELS.replace('labels_','').replace('.csv','').replace('_',' ').title()
    FEATURE_COLS = [c for c in df_desc_raw.columns
                    if c not in ['molecule_chembl_id','std_smiles',
                                 'pca_1','pca_2','tsne_1','tsne_2','umap_1','umap_2']]                    if df_desc_raw is not None else []
else:
    df_labels, X_fp, FEATURE_COLS = descargar_egfr_demo()
    df_desc_raw = df_labels
    TARGET_NAME = 'EGFR (demo)'

# Alinear y limpiar
if 'pActividad_mediana' in df_labels.columns and 'pActividad' not in df_labels.columns:
    df_labels = df_labels.rename(columns={'pActividad_mediana':'pActividad'})

df_labels = df_labels.dropna(subset=['activo']).reset_index(drop=True)
df_labels['activo'] = df_labels['activo'].astype(int)

print(f"\nDataset: {TARGET_NAME}")
print(f"  Moléculas:   {len(df_labels)}")
print(f"  Activos:     {df_labels['activo'].sum()} ({df_labels['activo'].mean()*100:.1f}%)")
print(f"  Inactivos:   {(df_labels['activo']==0).sum()} ({(1-df_labels['activo'].mean())*100:.1f}%)")


In [ ]:
# ── Construir matrices X e y ─────────────────────────────────────────────────
# Usamos descriptores fisicoquímicos como features principales.
# En las secciones siguientes también usaremos fingerprints de Morgan.

# y: etiqueta binaria (0=inactivo, 1=activo)
y = df_labels['activo'].values

# X_desc: descriptores fisicoquímicos (curados en NB-DATA-03)
if FEATURE_COLS and df_desc_raw is not None:
    cols_numericas = [c for c in FEATURE_COLS
                      if c in df_desc_raw.columns
                      and pd.api.types.is_numeric_dtype(df_desc_raw[c])]
    X_desc = df_desc_raw[cols_numericas].fillna(df_desc_raw[cols_numericas].median()).values
    NOMBRES_FEATURES = cols_numericas
else:
    # fallback: descriptores del df_labels
    cols_desc = ['MolWt','MolLogP','NumHDonors','NumHAcceptors',
                 'TPSA','NumRotatableBonds','RingCount']
    cols_desc = [c for c in cols_desc if c in df_labels.columns]
    X_desc = df_labels[cols_desc].fillna(df_labels[cols_desc].median()).values
    NOMBRES_FEATURES = cols_desc

print(f"Matriz de descriptores:   {X_desc.shape}")
if X_fp is not None:
    print(f"Matriz de fingerprints:   {X_fp.shape}")
print(f"Vector de etiquetas:      {y.shape}")
print(f"Features disponibles:     {NOMBRES_FEATURES[:10]}{'...' if len(NOMBRES_FEATURES)>10 else ''}")


In [ ]:
# ── División train / test estratificada ─────────────────────────────────────
# Estratificada = misma proporción de activos/inactivos en ambos conjuntos
# test_size=0.2 → 80% entrenamiento, 20% prueba

X_train_d, X_test_d, y_train, y_test = train_test_split(
    X_desc, y, test_size=0.2, random_state=42, stratify=y
)

if X_fp is not None:
    X_train_fp, X_test_fp, _, _ = train_test_split(
        X_fp, y, test_size=0.2, random_state=42, stratify=y
    )

print("DIVISIÓN TRAIN / TEST")
print("=" * 45)
print(f"  Train: {len(y_train):>5} muestras  "
      f"({y_train.sum()} activos, {(y_train==0).sum()} inactivos)")
print(f"  Test:  {len(y_test):>5} muestras  "
      f"({y_test.sum()} activos, {(y_test==0).sum()} inactivos)")
print()
print(f"  % activos en train: {y_train.mean()*100:.1f}%")
print(f"  % activos en test:  {y_test.mean()*100:.1f}%")
print()
print("✅ División estratificada — misma proporción en ambos conjuntos")


---
## 2. Baseline: ¿qué tan difícil es este problema?

Antes de entrenar cualquier modelo real, necesitamos un **baseline** (línea de base):  
el rendimiento que obtendríamos con la estrategia más tonta posible.

Si nuestro modelo no supera el baseline, **algo está mal**.


In [ ]:
# ── Métricas que usaremos en todo el notebook ────────────────────────────────
def evaluar_modelo(nombre, y_true, y_pred, y_proba=None):
    """
    Calcula y muestra las métricas de clasificación más relevantes para QSAR.

    Métricas:
    - Accuracy: % de predicciones correctas (poco útil con clases desbalanceadas)
    - AUC-ROC:  área bajo la curva ROC (0.5=azar, 1.0=perfecto)
    - F1:       media armónica de precisión y recall (bueno para desbalance)
    - MCC:      Matthews Correlation Coefficient (-1=peor, 0=azar, 1=perfecto)
                el más robusto ante desbalance de clases
    """
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_proba) if y_proba is not None else None

    resultado = {'Modelo': nombre, 'Accuracy': acc, 'F1': f1, 'MCC': mcc}
    if auc is not None:
        resultado['AUC-ROC'] = auc

    return resultado

# ── Diccionario para guardar resultados de todos los modelos ─────────────────
resultados_todos = []

# ── Baseline: siempre predecir la clase más frecuente ───────────────────────
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_d, y_train)
y_pred_dummy = dummy.predict(X_test_d)

res_dummy = evaluar_modelo('Baseline (mayoritaria)', y_test, y_pred_dummy)
resultados_todos.append(res_dummy)

print("BASELINE — Estrategia: siempre predecir clase mayoritaria")
print("=" * 55)
clase_mayor = 'Activo' if y_train.mean() > 0.5 else 'Inactivo'
print(f"  Clase mayoritaria en train: {clase_mayor}")
print()
for metrica, valor in res_dummy.items():
    if metrica != 'Modelo':
        print(f"  {metrica:<12}: {valor:.4f}")
print()
print("💡 Cualquier modelo real debe SUPERAR estos valores.")
print("   Un AUC-ROC cercano a 0.5 = el modelo no es mejor que el azar.")


---
## 3. Regresión Logística

La regresión logística es el modelo de clasificación más simple e interpretable.  
Aprende un **hiperplano** que separa las clases en el espacio de features.  

Aunque es "lineal", es un buen punto de partida porque:
- Es rápida de entrenar
- Sus coeficientes son directamente interpretables
- Generaliza bien con pocos datos


In [ ]:
# ── Pipeline: escalado + regresión logística ─────────────────────────────────
# Importante: la regresión logística y SVM son sensibles a la escala
# Por eso usamos un Pipeline que escala automáticamente antes de entrenar

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),           # Paso 1: escalar a media=0, std=1
    ('modelo', LogisticRegression(          # Paso 2: entrenar el modelo
        C=1.0,                              # C = inverso de la regularización
        max_iter=1000,
        class_weight='balanced',            # compensa clases desbalanceadas
        random_state=42
    ))
])

pipeline_lr.fit(X_train_d, y_train)

# Predicciones
y_pred_lr   = pipeline_lr.predict(X_test_d)
y_proba_lr  = pipeline_lr.predict_proba(X_test_d)[:, 1]

res_lr = evaluar_modelo('Regresión Logística', y_test, y_pred_lr, y_proba_lr)
resultados_todos.append(res_lr)

print("REGRESIÓN LOGÍSTICA")
print("=" * 45)
for metrica, valor in res_lr.items():
    if metrica != 'Modelo':
        mejora = valor - res_dummy.get(metrica, 0)
        signo = '+' if mejora > 0 else ''
        print(f"  {metrica:<12}: {valor:.4f}  ({signo}{mejora:.4f} vs baseline)")


In [ ]:
# ── Coeficientes: qué features importan en la regresión logística ────────────
coefs = pipeline_lr.named_steps['modelo'].coef_[0]
df_coefs = pd.DataFrame({
    'Feature': NOMBRES_FEATURES[:len(coefs)],
    'Coeficiente': coefs
}).sort_values('Coeficiente', key=abs, ascending=False)

print("COEFICIENTES DE LA REGRESIÓN LOGÍSTICA")
print("(valor absoluto mayor = feature más importante)")
print("=" * 55)
for _, row in df_coefs.head(10).iterrows():
    signo = '▲' if row['Coeficiente'] > 0 else '▼'
    barra = '█' * min(int(abs(row['Coeficiente']) * 5), 20)
    print(f"  {signo} {row['Feature']:<25} {row['Coeficiente']:>+8.4f}  {barra}")

print()
print("▲ Coeficiente positivo → aumentar este feature → más probabilidad de ACTIVO")
print("▼ Coeficiente negativo → aumentar este feature → más probabilidad de INACTIVO")


In [ ]:
# ── Matriz de confusión ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Inactivo', 'Activo'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusión — Regresión Logística', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrix_lr.png', dpi=120, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Verdaderos Negativos (TN): {tn}  — Inactivos bien clasificados")
print(f"Falsos Positivos    (FP): {fp}  — Inactivos clasificados como activos")
print(f"Falsos Negativos    (FN): {fn}  — Activos clasificados como inactivos")
print(f"Verdaderos Positivos (TP): {tp}  — Activos bien clasificados")
print()
sensibilidad = tp / (tp + fn) if (tp+fn) > 0 else 0
especificidad = tn / (tn + fp) if (tn+fp) > 0 else 0
print(f"Sensibilidad (Recall):  {sensibilidad:.4f}  — % activos correctamente identificados")
print(f"Especificidad:          {especificidad:.4f}  — % inactivos correctamente identificados")


---
## 4. Support Vector Machine (SVM)

La SVM busca el **hiperplano de margen máximo**: la frontera de decisión que maximiza  
la distancia entre las clases. Con el **kernel RBF** puede capturar relaciones no lineales.

**Hiperparámetros clave:**
- `C`: regularización — valores altos = más complejo, más riesgo de sobreajuste
- `gamma`: alcance del kernel — valores altos = solo vecinos cercanos influyen


In [ ]:
# ── Pipeline: escalado + SVM con kernel RBF ──────────────────────────────────
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', SVC(
        kernel='rbf',
        C=10.0,
        gamma='scale',           # gamma = 1/(n_features * X.var())
        probability=True,        # necesario para obtener probabilidades
        class_weight='balanced',
        random_state=42
    ))
])

print("Entrenando SVM (puede tardar 30-60 segundos en datasets grandes)...")
pipeline_svm.fit(X_train_d, y_train)

y_pred_svm  = pipeline_svm.predict(X_test_d)
y_proba_svm = pipeline_svm.predict_proba(X_test_d)[:, 1]

res_svm = evaluar_modelo('SVM (kernel RBF)', y_test, y_pred_svm, y_proba_svm)
resultados_todos.append(res_svm)

print()
print("SVM (kernel RBF)")
print("=" * 45)
for metrica, valor in res_svm.items():
    if metrica != 'Modelo':
        delta = valor - res_lr.get(metrica, 0)
        signo = '+' if delta > 0 else ''
        print(f"  {metrica:<12}: {valor:.4f}  ({signo}{delta:.4f} vs Reg. Logística)")


In [ ]:
# ── Efecto del parámetro C en la SVM ────────────────────────────────────────
valores_C = [0.01, 0.1, 1, 10, 100]
aucs_C = []

print("Efecto del hiperparámetro C sobre AUC-ROC (validación cruzada):")
print("=" * 50)
for C_val in valores_C:
    pipe_temp = Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', SVC(kernel='rbf', C=C_val, probability=True,
                       class_weight='balanced', random_state=42))
    ])
    scores = cross_val_score(pipe_temp, X_train_d, y_train,
                             cv=3, scoring='roc_auc', n_jobs=-1)
    aucs_C.append(scores.mean())
    print(f"  C={C_val:<8} AUC = {scores.mean():.4f} ± {scores.std():.4f}")

mejor_C = valores_C[np.argmax(aucs_C)]
print(f"\n  → Mejor C: {mejor_C}")


---
## 5. Random Forest

El Random Forest es un **ensemble de árboles de decisión** entrenados sobre  
subconjuntos aleatorios de datos y features. Es el modelo más usado en QSAR porque:
- Maneja bien features irrelevantes
- Robusto al sobreajuste
- Proporciona importancia de features directamente
- No requiere escalado de los datos


In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
# No necesita Pipeline con scaler — los árboles son invariantes a la escala

rf = RandomForestClassifier(
    n_estimators=500,        # número de árboles
    max_depth=None,          # árboles completos
    min_samples_leaf=2,      # mínimo de muestras en una hoja
    max_features='sqrt',     # features por árbol = sqrt(total)
    class_weight='balanced', # compensar desbalance
    n_jobs=-1,               # usar todos los núcleos disponibles
    random_state=42
)

print("Entrenando Random Forest (500 árboles)...")
rf.fit(X_train_d, y_train)

y_pred_rf  = rf.predict(X_test_d)
y_proba_rf = rf.predict_proba(X_test_d)[:, 1]

res_rf = evaluar_modelo('Random Forest', y_test, y_pred_rf, y_proba_rf)
resultados_todos.append(res_rf)

print()
print("RANDOM FOREST")
print("=" * 45)
for metrica, valor in res_rf.items():
    if metrica != 'Modelo':
        delta = valor - res_svm.get(metrica, 0)
        signo = '+' if delta > 0 else ''
        print(f"  {metrica:<12}: {valor:.4f}  ({signo}{delta:.4f} vs SVM)")


In [ ]:
# ── Importancia de features del Random Forest ────────────────────────────────
# El RF calcula automáticamente qué features son más útiles para las divisiones

importancias = rf.feature_importances_
nombres_feat = NOMBRES_FEATURES[:len(importancias)]

df_imp = pd.DataFrame({
    'Feature':     nombres_feat,
    'Importancia': importancias
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(df_imp)*0.35)))
colors = ['#58a6ff' if i >= len(df_imp)-5 else '#3d6b99'
          for i in range(len(df_imp))]
bars = ax.barh(df_imp['Feature'], df_imp['Importancia'],
               color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Importancia (Gini)', fontsize=11)
ax.set_title(f'Importancia de Features — Random Forest\n{TARGET_NAME}', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, val in zip(bars[-5:], df_imp['Importancia'].values[-5:]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=120, bbox_inches='tight')
plt.show()

print("TOP 5 FEATURES MÁS IMPORTANTES:")
for _, row in df_imp.tail(5).sort_values('Importancia', ascending=False).iterrows():
    print(f"  {row['Feature']:<25}: {row['Importancia']:.4f}")


In [ ]:
# ── Random Forest sobre fingerprints de Morgan ───────────────────────────────
# Los fingerprints son la representación más común en QSAR moderno
# 2048 features binarios vs 7-40 descriptores

if X_fp is not None and len(X_fp) == len(y):
    rf_fp = RandomForestClassifier(
        n_estimators=500, class_weight='balanced',
        n_jobs=-1, random_state=42
    )
    rf_fp.fit(X_train_fp, y_train)
    y_pred_rf_fp  = rf_fp.predict(X_test_fp)
    y_proba_rf_fp = rf_fp.predict_proba(X_test_fp)[:, 1]

    res_rf_fp = evaluar_modelo('RF + Morgan FP', y_test, y_pred_rf_fp, y_proba_rf_fp)
    resultados_todos.append(res_rf_fp)

    print("RANDOM FOREST + FINGERPRINTS MORGAN")
    print("=" * 45)
    for metrica, valor in res_rf_fp.items():
        if metrica != 'Modelo':
            delta = valor - res_rf.get(metrica, 0)
            signo = '+' if delta > 0 else ''
            print(f"  {metrica:<12}: {valor:.4f}  ({signo}{delta:.4f} vs RF+descriptores)")
    print()
    mejor_rf = res_rf_fp if res_rf_fp.get('AUC-ROC',0) > res_rf.get('AUC-ROC',0) else res_rf
    mejor_rf_name = 'RF + Morgan FP' if res_rf_fp.get('AUC-ROC',0) > res_rf.get('AUC-ROC',0) else 'RF + Descriptores'
    print(f"  → Mejor RF: {mejor_rf_name}")
else:
    print("⚠️  Fingerprints no disponibles — usando solo descriptores")
    rf_fp = None


---
## 6. Gradient Boosting (XGBoost)

El Gradient Boosting construye árboles **secuencialmente**, donde cada árbol  
corrige los errores del anterior. XGBoost es la implementación más eficiente  
y es el modelo ganador en muchas competencias de ML sobre datos tabulares.


In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / y_train.sum()  # compensar desbalance

if XGB_OK:
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,   # compensa desbalance de clases
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    print("Entrenando XGBoost...")
    xgb_model.fit(X_train_d, y_train)
    y_pred_xgb  = xgb_model.predict(X_test_d)
    y_proba_xgb = xgb_model.predict_proba(X_test_d)[:, 1]
    res_xgb = evaluar_modelo('XGBoost', y_test, y_pred_xgb, y_proba_xgb)
    resultados_todos.append(res_xgb)
    print("\nXGBOOST")
    print("=" * 45)
    for metrica, valor in res_xgb.items():
        if metrica != 'Modelo':
            delta = valor - res_rf.get(metrica, 0)
            signo = '+' if delta > 0 else ''
            print(f"  {metrica:<12}: {valor:.4f}  ({signo}{delta:.4f} vs RF)")
else:
    print("⚠️  XGBoost no disponible — usando Gradient Boosting de sklearn")
    gb = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                    learning_rate=0.05, random_state=42)
    gb.fit(X_train_d, y_train)
    y_pred_xgb  = gb.predict(X_test_d)
    y_proba_xgb = gb.predict_proba(X_test_d)[:, 1]
    res_xgb = evaluar_modelo('Gradient Boosting', y_test, y_pred_xgb, y_proba_xgb)
    resultados_todos.append(res_xgb)
    print(f"  AUC-ROC: {res_xgb.get('AUC-ROC',0):.4f}")


---
## 7. Comparación de todos los modelos

Ahora comparamos todos los modelos entrenados con las mismas métricas  
sobre el mismo conjunto de prueba.


In [ ]:
# ── Tabla comparativa ────────────────────────────────────────────────────────
df_resultados = pd.DataFrame(resultados_todos)
df_resultados = df_resultados.set_index('Modelo')
df_resultados = df_resultados.sort_values('AUC-ROC', ascending=False)

print("COMPARACIÓN DE MODELOS — Dataset de prueba")
print("=" * 65)
print(df_resultados.round(4).to_string())
print()
mejor_modelo = df_resultados.index[0]
print(f"🏆 Mejor modelo (por AUC-ROC): {mejor_modelo}")
print(f"   AUC-ROC = {df_resultados.loc[mejor_modelo, 'AUC-ROC']:.4f}")


In [ ]:
# ── Gráfico comparativo de métricas ─────────────────────────────────────────
metricas_plot = [c for c in ['AUC-ROC','F1','MCC','Accuracy'] if c in df_resultados.columns]
n_met = len(metricas_plot)
modelos = df_resultados.index.tolist()
x = np.arange(len(modelos))
width = 0.8 / n_met

fig, ax = plt.subplots(figsize=(max(10, len(modelos)*1.8), 5))
colores = ['#58a6ff','#3fb950','#f78166','#d2a8ff']

for i, (metrica, color) in enumerate(zip(metricas_plot, colores)):
    vals = df_resultados[metrica].values
    bars = ax.bar(x + i*width - (n_met-1)*width/2, vals,
                  width, label=metrica, color=color, alpha=0.85,
                  edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='AUC azar')
ax.set_xticks(x)
ax.set_xticklabels(modelos, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Valor de la métrica', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title(f'Comparación de modelos QSAR — {TARGET_NAME}', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=130, bbox_inches='tight')
plt.show()


---
## 8. Validación cruzada estratificada

Un único train/test split puede dar resultados optimistas o pesimistas por azar.  
La **validación cruzada k-fold** divide los datos en k partes y entrena k veces,  
usando cada parte como test una vez. Produce una estimación más robusta y confiable.

$$\text{AUC}_{CV} = \frac{1}{k} \sum_{i=1}^{k} \text{AUC}_i$$


In [ ]:
# ── Validación cruzada estratificada (5-fold) ────────────────────────────────
# Estratificada: preserva la proporción activo/inactivo en cada fold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos_cv = [
    ('Reg. Logística',  pipeline_lr),
    ('SVM (RBF)',       pipeline_svm),
    ('Random Forest',   rf),
]
if XGB_OK:
    modelos_cv.append(('XGBoost', xgb_model))

resultados_cv = {}
print("VALIDACIÓN CRUZADA 5-FOLD ESTRATIFICADA (AUC-ROC)")
print("=" * 60)

for nombre, modelo in modelos_cv:
    scores = cross_val_score(modelo, X_desc, y, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    resultados_cv[nombre] = scores
    barra = '█' * int(scores.mean() * 20)
    print(f"  {nombre:<22} {scores.mean():.4f} ± {scores.std():.4f}  {barra}")
    print(f"  {'':22} Folds: {[f'{s:.3f}' for s in scores]}")
    print()


In [ ]:
# ── Cargar resultados de validación cruzada desde GitHub ────────────────────
import requests, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL_CV = (
    "https://raw.githubusercontent.com/FelPVic/curso_datascience/"
    f"main/files/cv_results_egfr_(chembl203).json"
)

r = requests.get(URL_CV, timeout=30)
datos_cv = json.loads(r.text)

resultados_cv = datos_cv['resultados']

print(f"Target:  {datos_cv['target']}")
print(f"Umbral:  pActividad ≥ {datos_cv['umbral_pact']}")
print(f"Splits:  {datos_cv['n_splits']}-fold  |  scoring: {datos_cv['scoring']}")
print(f"Train:   {datos_cv['n_train']}  |  Test: {datos_cv['n_test']}")
print()
print(f"{'Modelo':<22} {'AUC media':>10} {'± std':>8} {'min':>7} {'max':>7}")
print("=" * 58)
for nombre, res in datos_cv['resumen'].items():
    print(f"  {nombre:<20} {res['media']:>10.4f} {res['std']:>8.4f} "
          f"{res['min']:>7.4f} {res['max']:>7.4f}")

In [ ]:
# ── Boxplot de validación cruzada ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
nombres_cv = list(resultados_cv.keys())
datos_cv   = list(resultados_cv.values())

bp = ax.boxplot(datos_cv, labels=nombres_cv, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
colores_box = ['#3d6b99','#6e40c9','#3fb950','#f78166']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('Validación cruzada 5-fold — AUC-ROC por modelo', fontsize=11)
ax.set_xticklabels(nombres_cv, rotation=15, ha='right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('validacion_cruzada.png', dpi=120, bbox_inches='tight')
plt.show()

print("💡 La altura del box muestra la variabilidad entre folds.")
print("   Un box estrecho = modelo estable y reproducible.")
print("   Un box ancho    = modelo sensible a qué datos se usan para entrenar.")


---
## 9. Curvas ROC y Precision-Recall

### Curva ROC
Muestra la tasa de verdaderos positivos (sensibilidad) vs tasa de falsos positivos  
a diferentes umbrales de decisión. El **AUC-ROC** es el área bajo esta curva.

### Curva Precision-Recall
Más informativa cuando las clases están muy desbalanceadas.  
El **AP** (Average Precision) resume la curva.


In [ ]:
# ── Curvas ROC de todos los modelos ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores_roc = {
    'Reg. Logística': '#58a6ff',
    'SVM (RBF)':      '#6e40c9',
    'Random Forest':  '#3fb950',
    'RF + Morgan FP': '#d2a8ff',
    'XGBoost':        '#f78166',
    'Gradient Boosting': '#f78166',
}

modelos_roc = [
    ('Reg. Logística', y_proba_lr),
    ('SVM (RBF)',      y_proba_svm),
    ('Random Forest',  y_proba_rf),
]
if X_fp is not None and rf_fp is not None:
    modelos_roc.append(('RF + Morgan FP', y_proba_rf_fp))
if XGB_OK:
    modelos_roc.append(('XGBoost', y_proba_xgb))

# Panel izquierdo: curva ROC
ax1 = axes[0]
for nombre, y_proba in modelos_roc:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    color = colores_roc.get(nombre, '#cdd6f4')
    ax1.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{nombre} (AUC={auc:.3f})')

ax1.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Azar (AUC=0.5)')
ax1.set_xlabel('Tasa de Falsos Positivos (1-Especificidad)', fontsize=10)
ax1.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=10)
ax1.set_title('Curva ROC', fontsize=11)
ax1.legend(fontsize=8, loc='lower right')
ax1.set_xlim([0,1]); ax1.set_ylim([0,1.02])

# Panel derecho: curva Precision-Recall
ax2 = axes[1]
for nombre, y_proba in modelos_roc:
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    color = colores_roc.get(nombre, '#cdd6f4')
    ax2.plot(rec, prec, color=color, linewidth=2,
             label=f'{nombre} (AP={ap:.3f})')

baseline_pr = y_test.mean()
ax2.axhline(baseline_pr, color='gray', linestyle='--', linewidth=1,
            label=f'Baseline ({baseline_pr:.2f})')
ax2.set_xlabel('Recall (Sensibilidad)', fontsize=10)
ax2.set_ylabel('Precisión', fontsize=10)
ax2.set_title('Curva Precision-Recall', fontsize=11)
ax2.legend(fontsize=8, loc='upper right')
ax2.set_xlim([0,1]); ax2.set_ylim([0,1.02])

plt.suptitle(f'Evaluación completa de modelos — {TARGET_NAME}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('curvas_roc_pr.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Análisis del umbral de decisión ─────────────────────────────────────────
# Por defecto sklearn usa 0.5 como umbral, pero podemos optimizarlo

# Usar el mejor modelo (RF) para analizar el efecto del umbral
y_proba_best = y_proba_rf  # cambiar según el mejor modelo

umbrales = np.arange(0.1, 0.95, 0.05)
f1s, sensibs, especifs, mccs = [], [], [], []

for u in umbrales:
    y_pred_u = (y_proba_best >= u).astype(int)
    f1s.append(f1_score(y_test, y_pred_u, zero_division=0))
    sensibs.append(((y_pred_u==1) & (y_test==1)).sum() / max(y_test.sum(), 1))
    especifs.append(((y_pred_u==0) & (y_test==0)).sum() / max((y_test==0).sum(), 1))
    mccs.append(matthews_corrcoef(y_test, y_pred_u))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(umbrales, f1s,      'b-o', markersize=4, label='F1')
ax.plot(umbrales, sensibs,  'g-s', markersize=4, label='Sensibilidad')
ax.plot(umbrales, especifs, 'r-^', markersize=4, label='Especificidad')
ax.plot(umbrales, mccs,     'm-D', markersize=4, label='MCC')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='Umbral default=0.5')

mejor_u_f1 = umbrales[np.argmax(f1s)]
ax.axvline(mejor_u_f1, color='blue', linestyle=':', linewidth=1.5,
           label=f'Mejor umbral F1={mejor_u_f1:.2f}')
ax.set_xlabel('Umbral de decisión', fontsize=11)
ax.set_ylabel('Métrica', fontsize=11)
ax.set_title('Efecto del umbral de decisión — Random Forest', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('analisis_umbral.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Mejor umbral por F1: {mejor_u_f1:.2f}")
print(f"💡 En drug discovery, a veces preferimos mayor sensibilidad")
print(f"   (no perdernos activos) a costa de más falsos positivos.")


---
## 10. Interpretación: ¿qué aprendió el modelo?

La interpretabilidad es fundamental en drug discovery — necesitamos entender  
qué propiedades fisicoquímicas distinguen a los activos de los inactivos.

Usamos **Permutation Importance**: mide cuánto empeora el modelo si mezclamos  
aleatoriamente los valores de cada feature.


In [ ]:
# ── Permutation Importance (modelo-agnóstico) ────────────────────────────────
# Funciona con cualquier modelo, a diferencia del Gini del RF

print("Calculando Permutation Importance (puede tardar 1-2 min)...")
perm_imp = permutation_importance(
    rf, X_test_d, y_test,
    n_repeats=20,          # mezclar 20 veces para estabilidad
    random_state=42,
    scoring='roc_auc',
    n_jobs=-1
)

df_perm = pd.DataFrame({
    'Feature':     NOMBRES_FEATURES[:len(perm_imp.importances_mean)],
    'Importancia': perm_imp.importances_mean,
    'Std':         perm_imp.importances_std,
}).sort_values('Importancia', ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(df_perm)*0.35)))
ax.barh(df_perm['Feature'], df_perm['Importancia'],
        xerr=df_perm['Std'], color='#58a6ff', alpha=0.8,
        edgecolor='white', capsize=3)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Disminución en AUC-ROC al mezclar el feature', fontsize=10)
ax.set_title('Permutation Importance — Random Forest\n(mayor valor = feature más importante)',
             fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('permutation_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nFEATURES MÁS IMPORTANTES (Permutation):")
for _, row in df_perm.tail(5).sort_values('Importancia', ascending=False).iterrows():
    print(f"  {row['Feature']:<25}: {row['Importancia']:.4f} ± {row['Std']:.4f}")


In [ ]:
# ── Distribución de features clave por clase ─────────────────────────────────
# Visualizar cómo difieren los activos de los inactivos en cada feature

top5_features = df_perm.tail(5)['Feature'].tolist()[::-1]
n_feat = len(top5_features)

fig, axes = plt.subplots(1, n_feat, figsize=(3.5*n_feat, 4))
if n_feat == 1: axes = [axes]

for ax, feat in zip(axes, top5_features):
    idx_feat = NOMBRES_FEATURES.index(feat) if feat in NOMBRES_FEATURES else None
    if idx_feat is None or idx_feat >= X_test_d.shape[1]:
        continue
    datos_feat = X_test_d[:, idx_feat]
    ax.hist(datos_feat[y_test==0], bins=25, alpha=0.6, color='#e74c3c',
            label='Inactivo', density=True, edgecolor='white')
    ax.hist(datos_feat[y_test==1], bins=25, alpha=0.6, color='#27ae60',
            label='Activo',   density=True, edgecolor='white')
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('Densidad', fontsize=9)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Distribución de features importantes por clase', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('distribucion_features_por_clase.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 11. Y-scrambling: validación de que el modelo realmente aprendió

El **Y-scrambling** mezcla aleatoriamente las etiquetas `y` rompiendo cualquier  
relación real con los features `X`. Si el modelo sigue siendo bueno después del  
scrambling, **solo memorizó los datos** — no aprendió nada útil.

> Un modelo válido debe tener AUC-ROC **significativamente mayor** que el AUC-ROC con Y-scrambling.


In [ ]:
# ── Y-scrambling con train/test split (el método correcto) ───────────────────
np.random.seed(42)
N_REPETICIONES = 20  # repetir el scrambling N veces para estimar distribución

aucs_scrambling = []
modelo_test = RandomForestClassifier(
    n_estimators=100,  # menos árboles para ser más rápido
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

print(f"Ejecutando Y-scrambling ({N_REPETICIONES} repeticiones)...")
for i in range(N_REPETICIONES):
    y_scrambled = y_train.copy()
    np.random.shuffle(y_scrambled)

    modelo_test.fit(X_train_d, y_scrambled)
    y_proba_scr = modelo_test.predict_proba(X_test_d)[:, 1]
    auc_scr = roc_auc_score(y_test, y_proba_scr)
    aucs_scrambling.append(auc_scr)

auc_real = res_rf['AUC-ROC']
auc_scr_mean = np.mean(aucs_scrambling)
auc_scr_std  = np.std(aucs_scrambling)

print()
print("Y-SCRAMBLING — RESULTADOS")
print("=" * 50)
print(f"  AUC-ROC modelo real:       {auc_real:.4f}")
print(f"  AUC-ROC scrambling (media):{auc_scr_mean:.4f} ± {auc_scr_std:.4f}")
print(f"  AUC-ROC scrambling (máx):  {max(aucs_scrambling):.4f}")
print()
diferencia = auc_real - auc_scr_mean
print(f"  Diferencia:                {diferencia:.4f}")
if diferencia > 0.15:
    print("  ✅ El modelo aprendió patrones reales — gran diferencia vs scrambling")
elif diferencia > 0.05:
    print("  ⚠️  El modelo aprendió algo, pero la diferencia es moderada")
else:
    print("  ❌ Diferencia pequeña — revisar el modelo y los datos")


In [ ]:
# ── Visualización del Y-scrambling ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(aucs_scrambling, bins=12, color='#e74c3c', alpha=0.7,
        edgecolor='white', linewidth=0.5, label='Y-scrambling')
ax.axvline(auc_real, color='#3fb950', linewidth=2.5,
           label=f'Modelo real (AUC={auc_real:.3f})')
ax.axvline(auc_scr_mean, color='#e74c3c', linewidth=2, linestyle='--',
           label=f'Scrambling media (AUC={auc_scr_mean:.3f})')
ax.axvline(0.5, color='gray', linewidth=1, linestyle=':', alpha=0.5,
           label='Azar (AUC=0.5)')

ax.set_xlabel('AUC-ROC', fontsize=11)
ax.set_ylabel('Frecuencia', fontsize=11)
ax.set_title(f'Y-Scrambling — Validación del modelo\n({N_REPETICIONES} repeticiones)',
             fontsize=11)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('y_scrambling.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 12. Pipeline completo y guardado del modelo

El modelo final se empaqueta en un pipeline reproducible que incluye  
el preprocesamiento y puede aplicarse directamente a nuevas moléculas.


In [ ]:
# ── Seleccionar el mejor modelo ─────────────────────────────────────────────
mejor_nombre = df_resultados.index[0]
print(f"Mejor modelo: {mejor_nombre}")
print(f"AUC-ROC:      {df_resultados.loc[mejor_nombre, 'AUC-ROC']:.4f}")
print()

# El RF no necesita scaler, pero lo incluimos para uniformidad
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

# Re-entrenar con TODOS los datos (train + test)
pipeline_final.fit(X_desc, y)
print("✅ Modelo re-entrenado con el dataset completo")


In [ ]:
# ── Función de predicción para nuevas moléculas ─────────────────────────────
from rdkit.Chem import Descriptors

def predecir_actividad(smiles_lista, modelo_pipeline, nombres_features):
    """
    Predice la actividad biológica de una lista de moléculas nuevas.

    Parámetros
    ----------
    smiles_lista    : list[str] — SMILES de las moléculas a predecir
    modelo_pipeline : Pipeline  — modelo entrenado con scaler incluido
    nombres_features: list[str] — nombres de los features en el orden correcto

    Retorna
    -------
    df_pred : DataFrame con SMILES, probabilidad, clase predicha
    """
    desc_map = {
        'MolWt':             Descriptors.MolWt,
        'MolLogP':           Descriptors.MolLogP,
        'NumHDonors':        Descriptors.NumHDonors,
        'NumHAcceptors':     Descriptors.NumHAcceptors,
        'TPSA':              Descriptors.TPSA,
        'NumRotatableBonds': Descriptors.NumRotatableBonds,
        'RingCount':         Descriptors.RingCount,
        'qed':               lambda m: __import__('rdkit.Chem.QED', fromlist=['qed']).qed(m),
    }

    rows = []
    validos = []
    for smi in smiles_lista:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            fila = {}
            for feat in nombres_features:
                fn = desc_map.get(feat)
                try:
                    fila[feat] = fn(mol) if fn else np.nan
                except Exception:
                    fila[feat] = np.nan
            rows.append(fila)
            validos.append(smi)
        else:
            print(f"  ⚠️  SMILES inválido omitido: {smi[:40]}")

    if not rows:
        return pd.DataFrame()

    X_nuevas = pd.DataFrame(rows)[nombres_features].fillna(0).values
    probabilidades = modelo_pipeline.predict_proba(X_nuevas)[:, 1]
    clases = (probabilidades >= 0.5).astype(int)

    return pd.DataFrame({
        'SMILES':           validos,
        'Prob_Activo':      probabilidades.round(4),
        'Clase':            ['Activo' if c==1 else 'Inactivo' for c in clases],
        'Confianza':        [f'{max(p, 1-p)*100:.1f}%' for p in probabilidades],
    })

# ── Probar con moléculas conocidas ───────────────────────────────────────────
NUEVAS_MOLECULAS = [
    # Erlotinib (inhibidor de EGFR aprobado)
    "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1",
    # Gefitinib
    "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
    # Aspirina (no debería ser activa contra EGFR)
    "CC(=O)Oc1ccccc1C(=O)O",
    # Cafeína (tampoco debería serlo)
    "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
]

df_predicciones = predecir_actividad(NUEVAS_MOLECULAS, pipeline_final, NOMBRES_FEATURES)
print("PREDICCIONES PARA MOLÉCULAS NUEVAS")
print("=" * 65)
print(df_predicciones.to_string(index=False))


In [ ]:
# ── Guardar el modelo y el reporte ──────────────────────────────────────────
TARGET_SLUG = TARGET_NAME.lower().replace(' ','_').replace('/','_')[:30]

# Guardar modelo con pickle
archivo_modelo = f'modelo_qsar_{TARGET_SLUG}.pkl'
with open(archivo_modelo, 'wb') as f:
    pickle.dump({
        'pipeline':       pipeline_final,
        'nombres_features': NOMBRES_FEATURES,
        'target':         TARGET_NAME,
        'metricas':       df_resultados.to_dict(),
        'umbral_decision': 0.5,
    }, f)

# Guardar tabla de resultados
archivo_reporte = f'reporte_modelos_{TARGET_SLUG}.csv'
df_resultados.round(4).to_csv(archivo_reporte)

print("✅ ARCHIVOS GUARDADOS")
print("=" * 50)
for archivo in [archivo_modelo, archivo_reporte]:
    tam = os.path.getsize(archivo)/1024
    print(f"  {archivo:<45} ({tam:.1f} KB)")

print()
print("Cómo cargar y usar el modelo en el futuro:")
print()
print("  import pickle")
print(f"  with open('{archivo_modelo}', 'rb') as f:")
print("      guardado = pickle.load(f)")
print("  modelo   = guardado['pipeline']")
print("  features = guardado['nombres_features']")
print("  predicciones = predecir_actividad(nuevas_smiles, modelo, features)")


In [ ]:
# ── Reporte final de evaluación ──────────────────────────────────────────────
print("=" * 65)
print(f"REPORTE FINAL — CLASIFICACIÓN QSAR: {TARGET_NAME.upper()}")
print("=" * 65)
print()
print(f"  Dataset:")
print(f"    Total moléculas:       {len(y)}")
print(f"    Activos:               {y.sum()} ({y.mean()*100:.1f}%)")
print(f"    Inactivos:             {(y==0).sum()} ({(1-y.mean())*100:.1f}%)")
print()
print(f"  Mejores resultados (conjunto de prueba):")
for metrica in ['AUC-ROC','F1','MCC','Accuracy']:
    if metrica in df_resultados.columns:
        mejor_val  = df_resultados[metrica].max()
        mejor_mod  = df_resultados[metrica].idxmax()
        print(f"    {metrica:<12}: {mejor_val:.4f}  ({mejor_mod})")
print()
print(f"  Validación:")
print(f"    Y-scrambling AUC:      {auc_scr_mean:.4f} ± {auc_scr_std:.4f}")
print(f"    AUC real vs scrambling:{auc_real - auc_scr_mean:.4f} de diferencia")
validez = "✅ MODELO VÁLIDO" if (auc_real - auc_scr_mean) > 0.1 else "⚠️  REVISAR"
print(f"    Conclusión:            {validez}")
print()
print("=" * 65)


---
## ✅ Resumen del notebook

| Sección | Técnica | Concepto clave |
|---------|---------|---------------|
| **2. Baseline** | `DummyClassifier` | Referencia mínima — todo modelo debe superarlo |
| **3. Reg. Logística** | `LogisticRegression` | Coeficientes = importancia directa de features |
| **4. SVM** | `SVC(kernel='rbf')` | Margen máximo; sensible a la escala → siempre escalar |
| **5. Random Forest** | `RandomForestClassifier` | Ensemble; importancia Gini; robusto al sobreajuste |
| **6. XGBoost** | `XGBClassifier` | Boosting iterativo; usualmente el más preciso |
| **7. Comparación** | Tabla + gráfico | AUC-ROC, F1, MCC, Accuracy sobre el mismo test set |
| **8. Validación cruzada** | `StratifiedKFold` | Estimación robusta con k=5 |
| **9. Curvas ROC/PR** | `roc_curve`, `precision_recall_curve` | Evaluación a todos los umbrales |
| **10. Interpretación** | Permutation Importance | Features que realmente mueven el AUC |
| **11. Y-scrambling** | Mezcla de etiquetas | Demostrar que el modelo no memorizó |
| **12. Pipeline** | `pickle`, `Pipeline` | Modelo reproducible y listo para producción |

## 📅 Próximo notebook: Semana 5 — Redes Neuronales

Con los features de NB-DATA-03 y la evaluación aprendida aquí, entrenaremos:
- **Red neuronal densa** con PyTorch para predicción de bioactividad
- Comparación directa con los modelos clásicos de este notebook
- **Graph Neural Networks** — representar moléculas como grafos

---
*NB-ML-01 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*
